In [44]:
import numpy as np
import pandas as pd

from ddc import (audit_regimes, rounded, wide_moment_table, wide_contribution_table, calibration_ladder, bin_mult_calibrator, lin_mult_calibrator, 
                 prospective_repair_regimes, prospective_repair_summary)

In [45]:
house = pd.read_parquet('house_clean.parquet')

In [46]:
census = (house.assign(X=lambda d: np.log1p(d["ttl_receipts"].clip(lower=0)), Y=lambda d: d["vote_share"], ) .dropna(subset=["X", "Y"]))

In [47]:
regimes = {
    "Receipts >= $10k": lambda d: d["ttl_receipts"].ge(10_000),
    "Receipts >= $100k": lambda d: d["ttl_receipts"].ge(100_000),
    "Ruling parties": lambda d: d["party"].isin(["DEM", "REP"]),
    "Open seats": lambda d: d["incumbency"].eq("O"),
    "Ruling party open seats": lambda d: d["party"].isin(["DEM", "REP"]) & d["incumbency"].eq("O")
}

In [48]:
coef_tbl, moment_tbl, contrib_tbl, beta_diag_tbl = audit_regimes(census, x="X", y="Y", regimes=regimes)
rounded(coef_tbl)

,regime,N,n,f,alpha_P,beta_P,alpha_A,beta_A,actual_gap,approx_gap,approx_error,abs_approx_error,rel_approx_error,var_x_P,max_abs_rho,max_abs_z_srs,max_realized_deff
0,Receipts >= $10k,8122,7866,0.9685,-0.537,0.0786,-0.5223,0.0775,-0.0011,-0.0009,-0.0002,0.0002,0.1771,3.2107,0.4202,37.8696,1434.1039
1,Receipts >= $100k,8122,6327,0.7790,-0.537,0.0786,-0.1169,0.0489,-0.0297,-0.0064,-0.0232,0.0232,0.7831,3.2107,0.8578,77.3018,5975.5638
2,Ruling parties,8122,7426,0.9143,-0.537,0.0786,-0.3335,0.0645,-0.0141,-0.0107,-0.0034,0.0034,0.2420,3.2107,0.4938,44.4978,1980.0524
3,Open seats,8122,926,0.1140,-0.537,0.0786,-0.6247,0.0795,0.0009,-0.0008,0.0017,0.0017,1.9030,3.2107,0.1202,10.8329,117.3519
4,Ruling party open seats,8122,791,0.0974,-0.537,0.0786,-0.3397,0.0600,-0.0186,-0.0207,0.0021,0.0021,-0.1122,3.2107,0.0785,7.0783,50.1019


In [49]:
coef_view = coef_tbl[['regime', 'N', 'n', 'f', 'beta_P', 'beta_A', 'actual_gap', 'approx_gap', 'approx_error', 'max_abs_rho', 'max_abs_z_srs']]
rounded(coef_view)

,regime,N,n,f,beta_P,beta_A,actual_gap,approx_gap,approx_error,max_abs_rho,max_abs_z_srs
0,Receipts >= $10k,8122,7866,0.9685,0.0786,0.0775,-0.0011,-0.0009,-0.0002,0.4202,37.8696
1,Receipts >= $100k,8122,6327,0.7790,0.0786,0.0489,-0.0297,-0.0064,-0.0232,0.8578,77.3018
2,Ruling parties,8122,7426,0.9143,0.0786,0.0645,-0.0141,-0.0107,-0.0034,0.4938,44.4978
3,Open seats,8122,926,0.1140,0.0786,0.0795,0.0009,-0.0008,0.0017,0.1202,10.8329
4,Ruling party open seats,8122,791,0.0974,0.0786,0.0600,-0.0186,-0.0207,0.0021,0.0785,7.0783


In [50]:
rounded(wide_moment_table(moment_tbl, value="rho"))

rho,regime,X,XY,X^2,Y
0,Open seats,0.0142,-0.0997,0.0157,-0.1202
1,Receipts >= $100k,0.8578,0.7050,0.8368,0.6411
2,Receipts >= $10k,0.4202,0.3066,0.3859,0.2985
3,Ruling parties,0.3712,0.4545,0.3547,0.4938
4,Ruling party open seats,0.0785,-0.0184,0.0778,-0.0318


In [51]:
rounded(wide_moment_table(moment_tbl, value="delta"))

delta,regime,X,XY,X^2,Y
0,Open seats,0.0708,-0.8849,1.9462,-0.0706
1,Receipts >= $100k,0.8187,1.1954,19.7942,0.0720
2,Receipts >= $10k,0.1358,0.1761,3.0917,0.0114
3,Ruling parties,0.2036,0.4429,4.8228,0.0319
4,Ruling party open seats,0.4285,-0.1784,10.5120,-0.0204


In [52]:
rounded(wide_moment_table(moment_tbl, value="z_srs"), digits=3)

z_srs,regime,X,XY,X^2,Y
0,Open seats,1.278,-8.986,1.417,-10.833
1,Receipts >= $100k,77.302,63.536,75.413,57.774
2,Receipts >= $10k,37.870,27.633,34.777,26.904
3,Ruling parties,33.452,40.958,31.968,44.498
4,Ruling party open seats,7.078,-1.659,7.007,-2.869


In [53]:
rounded(wide_contribution_table(contrib_tbl))

component,regime,X,XY,X^2,Y
0,Open seats,0.0345,-0.2756,-0.0476,0.2879
1,Receipts >= $100k,0.3992,0.3723,-0.4846,-0.2934
2,Receipts >= $10k,0.0662,0.0548,-0.0757,-0.0463
3,Ruling parties,0.0993,0.1380,-0.1181,-0.1299
4,Ruling party open seats,0.2089,-0.0556,-0.2573,0.0833


In [54]:
beta_diag_view = beta_diag_tbl[['regime', 'N', 'n', 'f', 'actual_gap', 'linearized_gap', 'beta_remainder', 'rho_psi', 'z_beta_lin', 'z_beta_actual', 'z_beta_remainder', 
                                'actual_by_linear']]

rounded(beta_diag_view, digits=4)

,regime,N,n,f,actual_gap,linearized_gap,beta_remainder,rho_psi,z_beta_lin,z_beta_actual,z_beta_remainder,actual_by_linear
0,Receipts >= $10k,8122,7866,0.9685,-0.0011,-0.0009,-0.0002,-0.0604,-5.4446,-6.6163,-1.1716,1.2152
1,Receipts >= $100k,8122,6327,0.7790,-0.0297,-0.0064,-0.0232,-0.1503,-13.5418,-62.4409,-48.8990,4.6110
2,Ruling parties,8122,7426,0.9143,-0.0141,-0.0107,-0.0034,-0.4349,-39.1939,-51.7080,-12.5141,1.3193
3,Open seats,8122,926,0.1140,0.0009,-0.0008,0.0017,-0.0035,-0.3162,0.3502,0.6664,-1.1074
4,Ruling party open seats,8122,791,0.0974,-0.0186,-0.0207,0.0021,-0.0846,-7.6209,-6.8522,0.7687,0.8991


In [55]:
calib_tbl = calibration_ladder(census, x="X", y="Y", regimes=regimes)

rounded(calib_tbl[["regime", "calibration_step", "converged", "beta_P", "beta_A", "beta_w", "gap_unweighted", "gap_weighted", "abs_gap_reduction", "ess", "min_w", 
                   "max_w", "weight_ratio"]], 5)

,regime,calibration_step,converged,beta_P,beta_A,beta_w,gap_unweighted,gap_weighted,abs_gap_reduction,ess,min_w,max_w,weight_ratio
0,Receipts >= $10k,Unweighted,True,0.0786,0.07753,0.07753,-0.00106,-0.00106,0.00000,7866.00000,0.00013,0.00013,1.00000
1,Receipts >= $10k,Calibrate X,True,0.0786,0.07753,0.07953,-0.00106,0.00094,0.00013,7813.07673,0.00010,0.00015,1.48920
2,Receipts >= $10k,"Calibrate X, X^2",True,0.0786,0.07753,0.07703,-0.00106,-0.00157,-0.00051,7707.76301,0.00011,0.00023,2.01924
3,Receipts >= $10k,"Calibrate X, Y",True,0.0786,0.07753,0.07950,-0.00106,0.00090,0.00016,7812.87398,0.00010,0.00015,1.49032
4,Receipts >= $10k,"Calibrate X, Y, X^2, XY",True,0.0786,0.07753,0.07860,-0.00106,0.00000,0.00106,7697.76192,0.00010,0.00024,2.39613
5,Receipts >= $100k,Unweighted,True,0.0786,0.04893,0.04893,-0.02966,-0.02966,0.00000,6327.00000,0.00016,0.00016,1.00000
6,Receipts >= $100k,Calibrate X,True,0.0786,0.04893,0.10276,-0.02966,0.02416,0.00550,3305.73145,0.00001,0.00078,117.41308
7,Receipts >= $100k,"Calibrate X, X^2",True,0.0786,0.04893,0.06356,-0.02966,-0.01504,0.01463,602.49060,0.00004,0.02248,581.44457
8,Receipts >= $100k,"Calibrate X, Y",True,0.0786,0.04893,0.10219,-0.02966,0.02359,0.00607,3285.15325,0.00001,0.00084,119.85552
9,Receipts >= $100k,"Calibrate X, Y, X^2, XY",True,0.0786,0.04893,0.07860,-0.02966,0.00000,0.02966,426.69095,0.00000,0.02871,11351.68793


In [56]:
prospective = prospective_repair_regimes(
    df=census,
    x="X",
    y="Y",
    regimes=regimes,
    cycle_col="cycle",
    holdout_cycle=2022,
    terms=("X", "Y", "X^2", "XY"),
    train_cycles=None,
    n_bins=10,
    bin_transform="identity",
    bin_stat="mean",
    ridge=1e-6,
    clip_log_multiplier=10,
    maxiter=1000,
)

rounded( prospective_repair_summary(prospective["result"])[["regime", "method", "N_holdout", "n_holdout", "beta_P", "beta_A", "beta_w", "gap_unweighted", "gap_weighted", 
                                                            "abs_gap_reduction", "ess_ratio", "max_relative_weight"]], 5)

,regime,method,N_holdout,n_holdout,beta_P,beta_A,beta_w,gap_unweighted,gap_weighted,abs_gap_reduction,ess_ratio,max_relative_weight
0,Receipts >= $10k,binned_multiplier_mean,833,811,0.07391,0.07369,0.07359,-0.00022,-0.00032,-0.00010,0.98465,1.36086
1,Receipts >= $10k,linear_multiplier,833,811,0.07391,0.07369,0.07368,-0.00022,-0.00023,-0.00001,0.97755,1.86616
2,Receipts >= $100k,binned_multiplier_mean,833,658,0.07391,0.05924,0.06824,-0.01466,-0.00567,0.00900,0.37683,4.38358
3,Receipts >= $100k,linear_multiplier,833,658,0.07391,0.05924,0.06154,-0.01466,-0.01237,0.00230,0.00525,246.54023
4,Ruling parties,binned_multiplier_mean,833,773,0.07391,0.06090,0.06069,-0.01301,-0.01322,-0.00021,0.98119,1.38969
5,Ruling parties,linear_multiplier,833,773,0.07391,0.06090,0.07288,-0.01301,-0.00103,0.01198,0.91611,4.93612
6,Open seats,binned_multiplier_mean,833,97,0.07391,0.07011,0.07272,-0.00380,-0.00118,0.00262,0.97118,1.31735
7,Open seats,linear_multiplier,833,97,0.07391,0.07011,0.07218,-0.00380,-0.00172,0.00208,0.73159,3.58400
8,Ruling party open seats,binned_multiplier_mean,833,86,0.07391,0.05457,0.05666,-0.01933,-0.01725,0.00209,0.88372,1.96636
9,Ruling party open seats,linear_multiplier,833,86,0.07391,0.05457,0.07242,-0.01933,-0.00148,0.01785,0.68351,4.88577


In [ ]:
HOLDOUT = 2022

cycles = sorted(census["cycle"].dropna().unique())
tuning_cycles = [cycle for cycle in cycles if cycle != HOLDOUT]

print("Tuning cycles:", tuning_cycles)
print("Final holdout:", HOLDOUT)

Tuning cycles: [np.int64(2004), np.int64(2006), np.int64(2008), np.int64(2010), np.int64(2012), np.int64(2014), np.int64(2016), np.int64(2018), np.int64(2020)]
Final holdout: 2022


In [58]:
def add_cv_scores(result, ess_floor=0.25, ess_penalty=0.01, max_rel_cap=10, max_rel_penalty=0.001):
    out = result.copy()

    out["abs_gap_before"] = out["gap_unweighted"].abs()
    out["abs_gap_after"] = out["gap_weighted"].abs()
    out["abs_gap_reduction"] = out["abs_gap_before"] - out["abs_gap_after"]

    out["pct_abs_gap_reduction"] = np.where(
        out["abs_gap_before"] > 1e-12,
        100 * out["abs_gap_reduction"] / out["abs_gap_before"],
        np.nan,
    )

    out["ess_shortfall"] = (ess_floor - out["ess_ratio"]).clip(lower=0)
    out["max_rel_excess"] = (np.log(out["max_relative_weight"]) - np.log(max_rel_cap)).clip(lower=0)

    out["cv_score"] = (
        out["abs_gap_after"]
        + ess_penalty * out["ess_shortfall"]
        + max_rel_penalty * out["max_rel_excess"]
    )

    out["improved"] = out["abs_gap_reduction"] > 0

    return out

In [ ]:
def cv_linear_multiplier_grid_temporal(
    df,
    x,
    y,
    regimes,
    cycle_col,
    final_holdout,
    ridge_grid=(1e-6, 1e-4, 1e-2, 1e-1, 1.0),
    clip_grid=(3, 5, 7, 10),
    min_train_cycles=3,
    terms=("X", "Y", "X^2", "XY"),
    maxiter=1000,
    ):
    cycles = sorted(df[cycle_col].dropna().unique())
    validation_cycles = [cycle for cycle in cycles if cycle < final_holdout]

    rows = []
    errors = []

    for holdout_cycle in validation_cycles:
        train_cycles = [cycle for cycle in cycles if cycle < holdout_cycle]

        if len(train_cycles) < min_train_cycles:
            continue

        for ridge in ridge_grid:
            for clip_log_multiplier in clip_grid:
                for regime_name, regime_rule in regimes.items():
                    try:
                        out = lin_mult_calibrator(
                            df=df,
                            x=x,
                            y=y,
                            cycle_col=cycle_col,
                            holdout_cycle=holdout_cycle,
                            regime=regime_rule,
                            terms=terms,
                            train_cycles=train_cycles,
                            ridge=ridge,
                            clip_log_multiplier=clip_log_multiplier,
                            maxiter=maxiter,
                        )

                        result = out["result"].copy()
                        result.insert(0, "regime", regime_name)
                        result["cv_holdout"] = holdout_cycle
                        result["ridge"] = ridge
                        result["clip_log_multiplier"] = clip_log_multiplier

                        rows.append(result)

                    except Exception as e:
                        errors.append({
                            "method": "linear_multiplier",
                            "cv_holdout": holdout_cycle,
                            "regime": regime_name,
                            "ridge": ridge,
                            "clip_log_multiplier": clip_log_multiplier,
                            "error": str(e),
                        })

    result = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    result = add_cv_scores(result) if len(result) else result

    return result, pd.DataFrame(errors)

In [74]:
linear_cv, linear_cv_errors = cv_linear_multiplier_grid_temporal(df=census, x="X", y="Y", regimes=regimes, cycle_col="cycle", final_holdout=2022, 
                                                                 ridge_grid=(1e-6, 1e-4, 1e-2, 1e-1, 1.0), clip_grid=(3, 5, 7, 10), maxiter=1000)

In [69]:
linear_cv_summary = (
    linear_cv
    .groupby(["regime", "ridge", "clip_log_multiplier"], as_index=False)
    .agg(
        mean_abs_gap_before=("abs_gap_before", "mean"),
        mean_abs_gap_after=("abs_gap_after", "mean"),
        mean_abs_gap_reduction=("abs_gap_reduction", "mean"),
        median_abs_gap_reduction=("abs_gap_reduction", "median"),
        mean_pct_abs_gap_reduction=("pct_abs_gap_reduction", "mean"),
        improvement_rate=("improved", "mean"),
        mean_ess_ratio=("ess_ratio", "mean"),
        min_ess_ratio=("ess_ratio", "min"),
        max_relative_weight=("max_relative_weight", "max"),
        mean_cv_score=("cv_score", "mean"),
    )
    .sort_values(["regime", "mean_cv_score"])
)

In [73]:
best_linear_by_regime = (
    linear_cv_summary
    .sort_values(["regime", "mean_cv_score"])
    .groupby("regime", as_index=False)
    .head(1)
)

rounded(
    best_linear_by_regime[[
        "regime",
        "ridge",
        "clip_log_multiplier",
        "mean_abs_gap_before",
        "mean_abs_gap_after",
        "mean_abs_gap_reduction",
        "improvement_rate",
        "mean_ess_ratio",
        "min_ess_ratio",
        "max_relative_weight",
    ]],
    5,
)

,regime,ridge,clip_log_multiplier,mean_abs_gap_before,mean_abs_gap_after,mean_abs_gap_reduction,improvement_rate,mean_ess_ratio,min_ess_ratio,max_relative_weight
16,Open seats,1.00,3,0.00719,0.01012,-0.00293,0.33333,0.66824,0.50992,6.95448
32,Receipts >= $100k,0.10,3,0.02398,0.01002,0.01396,0.83333,0.07439,0.05913,32.72080
48,Receipts >= $10k,0.01,3,0.00142,0.00143,-0.00000,0.50000,0.97444,0.96916,2.22034
72,Ruling parties,0.10,3,0.01417,0.00326,0.01091,1.00000,0.90003,0.84204,6.54839
96,Ruling party open seats,1.00,3,0.01475,0.00978,0.00497,0.83333,0.67779,0.55253,5.36438


In [ ]:
open_linear_cv = linear_cv.query("regime == 'Open seats'").copy()

rounded(
    open_linear_cv
    .groupby(["ridge", "clip_log_multiplier"], as_index=False)
    .agg(
        mean_abs_gap_before=("abs_gap_before", "mean"),
        mean_abs_gap_after=("abs_gap_after", "mean"),
        mean_abs_gap_reduction=("abs_gap_reduction", "mean"),
        improvement_rate=("improved", "mean"),
        mean_ess_ratio=("ess_ratio", "mean"),
        min_ess_ratio=("ess_ratio", "min"),
        max_relative_weight=("max_relative_weight", "max"),
    ).sort_values("mean_abs_gap_after"),5)

,ridge,clip_log_multiplier,mean_abs_gap_before,mean_abs_gap_after,mean_abs_gap_reduction,improvement_rate,mean_ess_ratio,min_ess_ratio,max_relative_weight
16,1.0000,3,0.00719,0.01012,-0.00293,0.33333,0.66824,0.50992,6.95448
19,1.0000,10,0.00719,0.01012,-0.00293,0.33333,0.66823,0.50985,6.95448
17,1.0000,5,0.00719,0.01012,-0.00293,0.33333,0.66823,0.50985,6.95448
18,1.0000,7,0.00719,0.01012,-0.00293,0.33333,0.66823,0.50985,6.95448
12,0.1000,3,0.00719,0.01050,-0.00331,0.33333,0.66185,0.48425,6.90432
14,0.1000,7,0.00719,0.01051,-0.00332,0.33333,0.66182,0.48407,6.90432
15,0.1000,10,0.00719,0.01051,-0.00332,0.33333,0.66182,0.48407,6.90432
13,0.1000,5,0.00719,0.01051,-0.00332,0.33333,0.66182,0.48407,6.90432
8,0.0100,3,0.00719,0.01070,-0.00351,0.33333,0.65983,0.47499,6.88483
10,0.0100,7,0.00719,0.01072,-0.00353,0.33333,0.65980,0.47479,6.88483


In [71]:
rounded(open_linear_cv.query('ridge == 1e-6 and clip_log_multiplier == 10')[['cv_holdout', 'beta_P', 'beta_A', 'beta_w', 'gap_unweighted', 'gap_weighted', 'abs_gap_reduction',
                                                                             'ess_ratio', 'max_relative_weight', ]], 5)

,cv_holdout,beta_P,beta_A,beta_w,gap_unweighted,gap_weighted,abs_gap_reduction,ess_ratio,max_relative_weight
18,2010,0.08903,0.10072,0.11439,0.01169,0.02536,-0.01367,0.47344,6.82089
118,2012,0.08058,0.07822,0.08080,-0.00235,0.00023,0.00213,0.82425,2.88234
218,2014,0.08048,0.08889,0.09100,0.00841,0.01052,-0.00211,0.59607,6.88216
318,2016,0.07712,0.07063,0.07699,-0.00649,-0.00013,0.00637,0.64371,4.81191
418,2018,0.07664,0.08118,0.08388,0.00455,0.00725,-0.00270,0.71578,4.87612
518,2020,0.06913,0.07877,0.09025,0.00965,0.02112,-0.01147,0.70378,3.70782


In [ ]:
# Open seats: restriction not stably repairable by this learned multiplier rule
# Ruling parties: stable, repairable defect
# Ruling parties open seats: partly repairable
# Receipts >= $100k: repairable with severe weight cost
# Receipts >= $10k: negligible defect, repair unnecessary